In [1]:
import sys
sys.path.append("..")
from datasets import load_dataset, load_from_disk
# from nltk.corpus import cmudict
import cmudict
import utils
from tqdm import tqdm

In [2]:
entries = cmudict.entries()
len(entries)

135166

In [3]:
dict_test = cmudict.dict()

In [4]:
dict_test["straightening"]

[['S', 'T', 'R', 'EY1', 'T', 'AH0', 'N', 'IH0', 'NG'],
 ['S', 'T', 'R', 'EY1', 'T', 'N', 'IH0', 'NG']]

In [5]:
# Create a reverse dictionary: phonemes -> list of words
phonemes_to_words = {}

def clean_phoneme(phonemes):
    cleaned = []
    for p in phonemes:
        cleaned.append(utils.PHONEME_TO_LOGIT[p.rstrip("012")])
    return cleaned


for word, phonemes in entries:
    # Convert list of phonemes to a tuple so it can be used as a dictionary key
    phoneme_key = tuple(clean_phoneme(phonemes))
    
    if phoneme_key not in phonemes_to_words:
        phonemes_to_words[phoneme_key] = []
    phonemes_to_words[phoneme_key].append(word)

print(f"Created reverse dictionary with {len(phonemes_to_words)} unique pronunciations.")

Created reverse dictionary with 114907 unique pronunciations.


In [6]:
data = utils.load_data(path='../data/t15_copyTask_neuralData/hdf5_data_final/')

100%|██████████| 45/45 [01:09<00:00,  1.54s/it]


In [6]:
data_ph = data[0][2]['seq_class_ids'][0]

def split_phoneme_sentence(p_sentence):
    p_chunks = []
    current_chunk = []

    for p in p_sentence:
        if p == 0: continue # Pad
        if p == 40:
            if current_chunk:
                p_chunks.append(tuple(current_chunk))
                current_chunk = []
        else:
            current_chunk.append(p)
    if current_chunk: p_chunks.append(tuple(current_chunk))
    return p_chunks

p_chunks = split_phoneme_sentence(data_ph)


NameError: name 'data' is not defined

In [8]:
data[0][2]['sentence_label'][0]

'You can see the code at this point as well.'

In [9]:
for p_word in p_chunks:
    print(phonemes_to_words.get(p_word, []))

['ewe', 'hugh', 'u', 'u.', 'uwe', 'yew', 'yoo', 'you', 'yu', 'yue']
['caen', 'cahn', 'can', 'cann', 'cannes', 'kan', 'kann', 'kanne']
['c', 'c.', 'cie', 'sci', 'sea', 'see', 'si', 'sie', 'sieh', 'tse']
['the', 'the']
['coad', 'code', 'coed']
['at']
['this', 'this', "this'"]
['point', 'pointe']
['as']
['well', 'welle']


In [10]:
# total = 0
# error = 0

# for session in data:
#     for p_sentence in session[1]['seq_class_ids']:
#         p_chunks = split_phoneme_sentence(p_sentence)
#         for p_word in p_chunks:
#             result = phonemes_to_words.get(p_word, [])
#             if not result:
#                 print(utils.indexes_to_phonemes(p_word))
#                 error += 1
#             # else:
#                 # print("Phonemes:", p_word, "-> Words:", result)
#             total += 1

# print(f"Total words: {total}, Errors: {error}, Error Rate: {error/total:.2%}")

In [11]:
from collections import defaultdict, Counter
import math

# 1. Build a simple Bigram Language Model from the training data
# We collect all sentences from the loaded data to train our LM
sentences = []
for session in data:
    for dataset in session:
        # Check if sentence_label exists and is not None
        if 'sentence_label' in dataset and dataset['sentence_label'] is not None:
            for s in dataset['sentence_label']:
                if s is None: continue
                if isinstance(s, bytes):
                    s = s.decode('utf-8')
                sentences.append(s.lower())

print(f"Training LM on {len(sentences)} sentences...")

# Count bigrams
bigram_counts = defaultdict(Counter)
unigram_counts = Counter()

for sentence in sentences:
    # Simple tokenization by splitting on space
    # For better results, consider using nltk.word_tokenize if available
    tokens = ['<s>'] + sentence.split() + ['</s>']
    for i in range(len(tokens) - 1):
        w1, w2 = tokens[i], tokens[i+1]
        bigram_counts[w1][w2] += 1
        unigram_counts[w1] += 1
    unigram_counts[tokens[-1]] += 1 # Count end token

# Function to get probability P(w2 | w1)
def get_bigram_prob(w1, w2, alpha=0.01):
    # Simple Add-alpha smoothing to handle unseen bigrams
    count_w1w2 = bigram_counts[w1][w2]
    count_w1 = unigram_counts[w1]
    vocab_size = len(unigram_counts)
    
    # If w1 is unknown, we fall back to uniform or just alpha smoothing
    if count_w1 == 0:
        return alpha / (alpha * vocab_size)
        
    return (count_w1w2 + alpha) / (count_w1 + alpha * vocab_size)

print("LM Built.")

Training LM on 8477 sentences...
LM Built.


In [12]:
def decode_sequence(phoneme_chunks):
    """
    Find the most likely sequence of words given a list of phoneme chunks using Viterbi algorithm.
    """
    # 1. Get candidates for each chunk
    candidates_list = []
    for p_chunk in phoneme_chunks:
        words = phonemes_to_words.get(p_chunk, [])
        if not words:
            # If no candidate found, use a placeholder to allow decoding to continue
            # In a real system, you might use a phoneme-to-grapheme model here
            words = ["<UNK>"]
        candidates_list.append(words)
        
    # 2. Viterbi Algorithm
    T = len(candidates_list)
    if T == 0: return []
    
    # dp[t][word] = log_prob of best path ending at 'word' at step 't'
    dp = [{} for _ in range(T)]
    # backpointer[t][word] = previous word that yielded the max prob
    backpointer = [{} for _ in range(T)]
    
    # Initialization (t=0)
    for word in candidates_list[0]:
        # P(word | <s>)
        prob = get_bigram_prob('<s>', word)
        dp[0][word] = math.log(prob)
        backpointer[0][word] = None
        
    # Recursion (t=1 to T-1)
    for t in range(1, T):
        for word in candidates_list[t]:
            best_prev_log_prob = -float('inf')
            best_prev_word = None
            
            for prev_word in candidates_list[t-1]:
                # P(word | prev_word)
                trans_prob = get_bigram_prob(prev_word, word)
                prev_log_prob = dp[t-1][prev_word]
                
                total_log_prob = prev_log_prob + math.log(trans_prob)
                
                if total_log_prob > best_prev_log_prob:
                    best_prev_log_prob = total_log_prob
                    best_prev_word = prev_word
            
            if best_prev_word is not None:
                dp[t][word] = best_prev_log_prob
                backpointer[t][word] = best_prev_word
            else:
                # Fallback if no path found (should be rare with smoothing)
                dp[t][word] = -float('inf')
                # Just pick the first one from previous to keep chain
                backpointer[t][word] = candidates_list[t-1][0]

    # Termination
    best_final_log_prob = -float('inf')
    best_last_word = None
    
    for word in candidates_list[T-1]:
        # Optional: Multiply by P(</s> | word)
        end_prob = get_bigram_prob(word, '</s>')
        total_prob = dp[T-1][word] + math.log(end_prob)
        
        if total_prob > best_final_log_prob:
            best_final_log_prob = total_prob
            best_last_word = word
            
    # Backtrack
    if best_last_word is None:
        # If something went wrong, just pick the first candidate for each
        return [c[0] for c in candidates_list]

    best_path = []
    current_word = best_last_word
    
    for t in range(T-1, -1, -1):
        best_path.append(current_word)
        current_word = backpointer[t][current_word]
        
    return best_path[::-1]

# Test on the example
print("Phoneme Chunks:", p_chunks)

# Get original sentence
original_sentence = data[0][2]['sentence_label'][0]
if isinstance(original_sentence, bytes):
    original_sentence = original_sentence.decode('utf-8')

decoded_sentence = decode_sequence(p_chunks)

print(f"Original Sentence: {original_sentence}")
print(f"Decoded Sentence:  {' '.join(decoded_sentence)}")
print("-" * 40)

Phoneme Chunks: [(np.int32(37), np.int32(34)), (np.int32(20), np.int32(2), np.int32(23)), (np.int32(29), np.int32(18)), (np.int32(10), np.int32(3)), (np.int32(20), np.int32(25), np.int32(9)), (np.int32(2), np.int32(31)), (np.int32(10), np.int32(17), np.int32(29)), (np.int32(27), np.int32(26), np.int32(23), np.int32(31)), (np.int32(2), np.int32(38)), (np.int32(36), np.int32(11), np.int32(21))]
Original Sentence: You can see the code at this point as well.
Decoded Sentence:  you can see the code at this point as well
----------------------------------------


In [13]:
def levenshtein_distance(s1, s2):
    """Calculates the Levenshtein distance between two sequences."""
    if len(s1) < len(s2):
        return levenshtein_distance(s2, s1)

    if len(s2) == 0:
        return len(s1)

    previous_row = range(len(s2) + 1)
    for i, c1 in enumerate(s1):
        current_row = [i + 1]
        for j, c2 in enumerate(s2):
            insertions = previous_row[j + 1] + 1
            deletions = current_row[j] + 1
            substitutions = previous_row[j] + (c1 != c2)
            current_row.append(min(insertions, deletions, substitutions))
        previous_row = current_row
    
    return previous_row[-1]

def get_fuzzy_candidates(phoneme_seq, max_distance=2):
    """
    Finds dictionary entries that are close to the input phoneme sequence.
    Returns a list of words and the distance found.
    """
    candidates = []
    min_dist = float('inf')
    
    # Optimization: Only check keys with length close to input
    target_len = len(phoneme_seq)
    
    # Iterate through all known pronunciations
    for key in phonemes_to_words:
        # Skip if length difference is already too big
        if abs(len(key) - target_len) > max_distance:
            continue
            
        dist = levenshtein_distance(phoneme_seq, key)
        
        if dist < min_dist:
            min_dist = dist
            candidates = phonemes_to_words[key]
        elif dist == min_dist:
            candidates.extend(phonemes_to_words[key])
            
    if min_dist <= max_distance:
        return list(set(candidates)) # Remove duplicates
    return []

print("Fuzzy matching functions defined.")

Fuzzy matching functions defined.


In [14]:
def decode_sequence_with_fuzzy(phoneme_chunks):
    """
    Find the most likely sequence of words, using fuzzy matching for unseen phoneme sequences.
    """
    # 1. Get candidates for each chunk
    candidates_list = []
    for p_chunk in phoneme_chunks:
        # Try exact match
        words = phonemes_to_words.get(p_chunk, [])
        
        # If no exact match, try fuzzy match
        if not words:
            words = get_fuzzy_candidates(p_chunk, max_distance=2)
            
        if not words:
            # If still no candidate, use placeholder
            words = ["<UNK>"]
            
        candidates_list.append(words)
        
    # 2. Viterbi Algorithm (Same as before)
    T = len(candidates_list)
    if T == 0: return []
    
    dp = [{} for _ in range(T)]
    backpointer = [{} for _ in range(T)]
    
    # Initialization (t=0)
    for word in candidates_list[0]:
        prob = get_bigram_prob('<s>', word)
        dp[0][word] = math.log(prob)
        backpointer[0][word] = None
        
    # Recursion
    for t in range(1, T):
        for word in candidates_list[t]:
            best_prev_log_prob = -float('inf')
            best_prev_word = None
            
            for prev_word in candidates_list[t-1]:
                trans_prob = get_bigram_prob(prev_word, word)
                prev_log_prob = dp[t-1][prev_word]
                total_log_prob = prev_log_prob + math.log(trans_prob)
                
                if total_log_prob > best_prev_log_prob:
                    best_prev_log_prob = total_log_prob
                    best_prev_word = prev_word
            
            if best_prev_word is not None:
                dp[t][word] = best_prev_log_prob
                backpointer[t][word] = best_prev_word
            else:
                dp[t][word] = -float('inf')
                backpointer[t][word] = candidates_list[t-1][0]

    # Termination
    best_final_log_prob = -float('inf')
    best_last_word = None
    
    for word in candidates_list[T-1]:
        end_prob = get_bigram_prob(word, '</s>')
        total_prob = dp[T-1][word] + math.log(end_prob)
        
        if total_prob > best_final_log_prob:
            best_final_log_prob = total_prob
            best_last_word = word
            
    # Backtrack
    if best_last_word is None:
        return [c[0] for c in candidates_list]

    best_path = []
    current_word = best_last_word
    
    for t in range(T-1, -1, -1):
        best_path.append(current_word)
        current_word = backpointer[t][current_word]
        
    return best_path[::-1]

# Test
print("Phoneme Chunks:", p_chunks)
decoded_sentence = decode_sequence_with_fuzzy(p_chunks)

print("-" * 40)
print(f"Original Sentence: {original_sentence}")
print(f"Decoded Sentence:  {' '.join(decoded_sentence)}")
print("-" * 40)

Phoneme Chunks: [(np.int32(37), np.int32(34)), (np.int32(20), np.int32(2), np.int32(23)), (np.int32(29), np.int32(18)), (np.int32(10), np.int32(3)), (np.int32(20), np.int32(25), np.int32(9)), (np.int32(2), np.int32(31)), (np.int32(10), np.int32(17), np.int32(29)), (np.int32(27), np.int32(26), np.int32(23), np.int32(31)), (np.int32(2), np.int32(38)), (np.int32(36), np.int32(11), np.int32(21))]
----------------------------------------
Original Sentence: You can see the code at this point as well.
Decoded Sentence:  you can see the code at this point as well
----------------------------------------


In [20]:
import numpy as np
import re

def calculate_wer(reference, hypothesis):
    """
    Calculate Word Error Rate (WER).
    WER = (Substitutions + Insertions + Deletions) / Number of Words in Reference
    """
    ref_words = reference.split()
    hyp_words = hypothesis.split()
    
    # Reuse the levenshtein_distance function defined earlier
    # It works on lists of strings as well as strings!
    dist = levenshtein_distance(ref_words, hyp_words)
    
    if len(ref_words) == 0:
        return 0.0 if len(hyp_words) == 0 else 1.0
    return dist / len(ref_words)

def evaluate_decoder(decoder_func, dataset, num_samples=50, show_errors=True):
    total_wer = 0
    count = 0
    
    # Use a subset for speed
    num_available = len(dataset['seq_class_ids'])
    indices = range(min(num_available, num_samples))
    
    print(f"Evaluating on {len(indices)} samples...")
    
    for i in indices:
        # Get inputs
        p_sentence = dataset['seq_class_ids'][i]
        original_sentence = dataset['sentence_label'][i]
        
        if original_sentence is None: continue
        if isinstance(original_sentence, bytes):
            original_sentence = original_sentence.decode('utf-8')
        original_sentence = original_sentence.lower()
        
        # Remove punctuation from Reference
        original_sentence = re.sub(r'[^\w\s\']', '', original_sentence)
            
        # Process
        p_chunks = split_phoneme_sentence(p_sentence)
        decoded_words = decoder_func(p_chunks)
        decoded_sentence = " ".join(decoded_words)
        
        # Remove punctuation from Hypothesis (some dictionary words might have it, e.g. "mr.")
        decoded_sentence = re.sub(r'[^\w\s\']', '', decoded_sentence)
        
        # Metric
        wer = calculate_wer(original_sentence, decoded_sentence)
        total_wer += wer
        count += 1
        
        if show_errors and wer > 0:
            print(f"\n[Sample {i}] WER: {wer:.2f}")
            print(f"REF: {original_sentence}")
            print(f"HYP: {decoded_sentence}")
            
            # Identify mismatched words (simple set difference for display)
            ref_set = set(original_sentence.split())
            hyp_set = set(decoded_sentence.split())
            missed = ref_set - hyp_set
            wrong = hyp_set - ref_set
            if missed: print(f"Missed: {missed}")
            if wrong:  print(f"Wrong/Added: {wrong}")
        
    avg_wer = total_wer / count if count > 0 else 0
    return avg_wer

# Select a dataset (e.g., validation set of first session)
# data[0][2] corresponds to the validation set of the first session
eval_data = data[0][2]

print("Evaluating Exact Match Decoder...")
# We suppress errors for the exact match to avoid clutter, but you can set show_errors=True
wer_exact = evaluate_decoder(decode_sequence, eval_data, show_errors=False)
print(f"Exact Match WER: {wer_exact:.2%}")

print("\nEvaluating Fuzzy Match Decoder (showing errors)...")
wer_fuzzy = evaluate_decoder(decode_sequence_with_fuzzy, eval_data, show_errors=True)
print(f"Fuzzy Match WER: {wer_fuzzy:.2%}")

Evaluating Exact Match Decoder...
Evaluating on 35 samples...
Exact Match WER: 2.55%

Evaluating Fuzzy Match Decoder (showing errors)...
Evaluating on 35 samples...

[Sample 19] WER: 0.25
REF: he does the yard
HYP: he does the yarde
Missed: {'yard'}
Wrong/Added: {'yarde'}

[Sample 22] WER: 0.14
REF: not for the job i have now
HYP: not for the job i have gnau
Missed: {'now'}
Wrong/Added: {'gnau'}

[Sample 27] WER: 0.17
REF: bacon and all that good stuff
HYP: bacon and all that good stough
Missed: {'stuff'}
Wrong/Added: {'stough'}

[Sample 28] WER: 0.25
REF: if you look back
HYP: if you look backe
Missed: {'back'}
Wrong/Added: {'backe'}

[Sample 30] WER: 0.08
REF: she came last june and watched a game in the sky dome
HYP: she came last june and watched a game in the sky dohme
Missed: {'dome'}
Wrong/Added: {'dohme'}
Fuzzy Match WER: 2.55%


In [23]:
def evaluate_on_all_validation_sets(decoder_func, data, num_samples_per_session=50):
    total_wer_sum = 0
    total_samples = 0
    
    print(f"Evaluating on {len(data)} sessions (Validation Sets)...")
    
    for session_idx, session_data in enumerate(data):
        # Index 2 is the validation set (data_val.hdf5)
        # Index 0 is test (no labels), Index 1 is train
        if len(session_data) < 3:
            print(f"Session {session_idx+1}: Missing validation set. Skipping.")
            continue
            
        val_set = session_data[2] 
        
        # Evaluate on this session
        session_wer = evaluate_decoder(decoder_func, val_set, num_samples=num_samples_per_session, show_errors=False)
        
        print(f"Session {session_idx+1} Val WER: {session_wer:.2%}")
        
        total_wer_sum += session_wer
        total_samples += 1
        
    global_avg_wer = total_wer_sum / total_samples if total_samples > 0 else 0
    return global_avg_wer

print("\n" + "="*40)
print("Global Evaluation on All Validation Sets (Fuzzy Decoder)")
print("="*40)

# Run evaluation
global_wer = evaluate_on_all_validation_sets(decode_sequence, data, num_samples_per_session=50)
print(f"\nAverage WER across all validation sets: {global_wer:.2%}")


Global Evaluation on All Validation Sets (Fuzzy Decoder)
Evaluating on 41 sessions (Validation Sets)...
Evaluating on 35 samples...
Session 1 Val WER: 2.55%
Evaluating on 49 samples...
Session 2 Val WER: 4.71%
Evaluating on 48 samples...
Session 3 Val WER: 5.40%
Evaluating on 25 samples...
Session 4 Val WER: 1.07%
Evaluating on 25 samples...
Session 5 Val WER: 4.48%
Evaluating on 49 samples...
Session 6 Val WER: 4.84%
Evaluating on 34 samples...
Session 7 Val WER: 4.59%
Evaluating on 35 samples...
Session 8 Val WER: 4.33%
Evaluating on 48 samples...
Session 9 Val WER: 2.64%
Evaluating on 44 samples...
Session 10 Val WER: 4.60%
Evaluating on 36 samples...
Session 11 Val WER: 5.49%
Evaluating on 17 samples...
Session 12 Val WER: 1.57%
Evaluating on 44 samples...
Session 13 Val WER: 3.03%
Evaluating on 44 samples...
Session 14 Val WER: 1.78%
Evaluating on 9 samples...
Session 15 Val WER: 2.82%
Evaluating on 33 samples...
Session 16 Val WER: 5.47%
Evaluating on 50 samples...
Session 17 Va